# NB02 - Bronze Data Validation

## Objective

This notebook validates the Bronze layer before any transformation.

Validation includes:

- Row count verification
- Null value analysis
- Duplicate primary key detection
- Schema verification

Successful validation confirms that the Bronze layer is ready for transformation into the Silver layer.

In [1]:
# ==========================================================
# Import Libraries
# ==========================================================

from pyspark.sql import functions as F

print("Libraries Loaded Successfully")

StatementMeta(, 8172b8d0-03f9-4ee1-aced-ad7ffbfacca0, 3, Finished, Available, Finished, False)

Libraries Loaded Successfully


## Bronze Tables

The following Bronze tables will be validated.

In [2]:
tables = [

    "DimRegion",
    "DimStore",
    "DimCustomer",
    "DimSupplier",
    "DimProduct",
    "DimEmployee",
    "DimPromotion",
    "DimDate",
    "FactSales",
    "FactInventory",
    "FactReturns"

]

StatementMeta(, 8172b8d0-03f9-4ee1-aced-ad7ffbfacca0, 4, Finished, Available, Finished, False)

## Row Count Validation

### Objective

Verify that every Bronze table contains data.

This confirms successful ingestion from the Parquet files.

In [3]:
print("=" * 70)
print("ROW COUNT VALIDATION")
print("=" * 70)

for table in tables:

    count = spark.table(table).count()

    print(f"{table:<20} {count:>12,}")

StatementMeta(, 8172b8d0-03f9-4ee1-aced-ad7ffbfacca0, 5, Finished, Available, Finished, False)

ROW COUNT VALIDATION
DimRegion                       6
DimStore                      100
DimCustomer                50,000
DimSupplier                   500
DimProduct                  5,000
DimEmployee                   500
DimPromotion                  100
DimDate                     2,191
FactSales               1,000,000
FactInventory             250,000
FactReturns                77,065


## Null Value Validation

### Objective

Identify null values in each table.

This helps detect incomplete or missing records before transformation.

In [4]:
for table in tables:

    print("\n" + "=" * 70)
    print(table)
    print("=" * 70)

    df = spark.table(table)

    null_df = df.select([

        F.sum(
            F.col(c).isNull().cast("int")
        ).alias(c)

        for c in df.columns

    ])

    null_df.show(truncate=False)

StatementMeta(, 8172b8d0-03f9-4ee1-aced-ad7ffbfacca0, 6, Finished, Available, Finished, False)


DimRegion
+---------+----------+-------+
|RegionKey|RegionName|Country|
+---------+----------+-------+
|0        |0         |0      |
+---------+----------+-------+


DimStore
+--------+---------+---------+----+-----+---------+---------+-----------+--------+
|StoreKey|StoreCode|StoreName|City|State|RegionKey|StoreType|OpeningDate|IsActive|
+--------+---------+---------+----+-----+---------+---------+-----------+--------+
|0       |0        |0        |0   |0    |0        |0        |0          |0       |
+--------+---------+---------+----+-----+---------+---------+-----------+--------+


DimCustomer
+-----------+----------+------------+------+-----------+---+--------+----+-----+---------+-----+-----+--------+-----------+
|CustomerKey|CustomerID|CustomerName|Gender|DateOfBirth|Age|AgeGroup|City|State|RegionKey|Email|Phone|JoinDate|LoyaltyTier|
+-----------+----------+------------+------+-----------+---+--------+----+-----+---------+-----+-----+--------+-----------+
|0          |0        

## Duplicate Primary Key Validation

### Objective

Verify that each table contains unique primary keys.

Duplicate keys indicate data integrity issues.

In [5]:
primary_keys = {

    "DimRegion": "RegionKey",
    "DimStore": "StoreKey",
    "DimCustomer": "CustomerKey",
    "DimSupplier": "SupplierKey",
    "DimProduct": "ProductKey",
    "DimEmployee": "EmployeeKey",
    "DimPromotion": "PromotionKey",
    "DimDate": "DateKey",
    "FactSales": "SalesKey",
    "FactInventory": "InventoryKey",
    "FactReturns": "ReturnKey"

}

for table, key in primary_keys.items():

    duplicates = (

        spark.table(table)

        .groupBy(key)

        .count()

        .filter("count > 1")

        .count()

    )

    print(f"{table:<20} Duplicate Keys : {duplicates}")

StatementMeta(, 8172b8d0-03f9-4ee1-aced-ad7ffbfacca0, 7, Finished, Available, Finished, False)

DimRegion            Duplicate Keys : 0
DimStore             Duplicate Keys : 0
DimCustomer          Duplicate Keys : 0
DimSupplier          Duplicate Keys : 0
DimProduct           Duplicate Keys : 0
DimEmployee          Duplicate Keys : 0
DimPromotion         Duplicate Keys : 0
DimDate              Duplicate Keys : 0
FactSales            Duplicate Keys : 0
FactInventory        Duplicate Keys : 0
FactReturns          Duplicate Keys : 0


# Validation Summary

## Validation Completed

The following checks were performed successfully:

- Row Count Validation
- Null Value Validation
- Duplicate Primary Key Validation

If no critical issues are found, the Bronze layer is approved for transformation to the Silver layer.